## Section 1 Import และ Path Setup

In [ ]:
!pip install pyedflib --quiet

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import glob
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyedflib

PROJECT_DIR = "/content/drive/MyDrive/SleepStageClassificationProject"
DATA_DIR = f"{PROJECT_DIR}/dataset/HMC"

EDA_OUTPUT_DIR = f"{PROJECT_DIR}/results/eda_results_new"
FIGURES_DIR = f"{EDA_OUTPUT_DIR}/figures"

os.makedirs(EDA_OUTPUT_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

print("DATA_DIR exists:", os.path.isdir(DATA_DIR))

In [ ]:
signal_files = sorted(f for f in glob.glob(os.path.join(DATA_DIR, "*.edf")) if "_sleepscoring" not in os.path.basename(f).lower())
scoring_edf_files = sorted(glob.glob(os.path.join(DATA_DIR, "*_sleepscoring.edf")))
scoring_txt_files = sorted(glob.glob(os.path.join(DATA_DIR, "*_sleepscoring.txt")))

signal_ids = [os.path.basename(f)[:-4] for f in signal_files]
scoring_edf_ids = [os.path.basename(f)[:-len("_sleepscoring.edf")] for f in scoring_edf_files]
scoring_txt_ids = [os.path.basename(f)[:-len("_sleepscoring.txt")] for f in scoring_txt_files]

print("Signal EDF:", len(signal_files), "| Scoring EDF:", len(scoring_edf_files),
      "| Scoring TXT:", len(scoring_txt_files))

all_ids = sorted(set(signal_ids) | set(scoring_edf_ids) | set(scoring_txt_ids))

recording_inventory = pd.DataFrame([{
    "subject_id": sid,
    "signal_exists": sid in signal_ids,
    "scoring_edf_exists": sid in scoring_edf_ids,
    "scoring_txt_exists": sid in scoring_txt_ids,
} for sid in all_ids])

for col in ["signal_exists", "scoring_edf_exists", "scoring_txt_exists"]:
    print(f"Missing {col.removesuffix('_exists')}:", int((~recording_inventory[col]).sum()))

recording_inventory.to_csv(os.path.join(EDA_OUTPUT_DIR, "recording_inventory.csv"), index=False)
display(recording_inventory.head(10))

## Section 2: Recording / File Integrity

In [ ]:
def read_scoring(path):
    df = pd.read_csv(path, sep=",", skipinitialspace=True)
    df.columns = df.columns.str.strip()
    required = ["Recording onset", "Duration", "Annotation"]
    if not set(required).issubset(df.columns):
        raise ValueError("Missing scoring columns")
    return df

signal_readable, scoring_edf_readable, scoring_txt_readable = [], [], []

for _, row in recording_inventory.iterrows():
    sid = row["subject_id"]
    paths = [
        (row["signal_exists"], os.path.join(DATA_DIR, sid + ".edf"), "edf"),
        (row["scoring_edf_exists"], os.path.join(DATA_DIR, sid + "_sleepscoring.edf"), "edf"),
        (row["scoring_txt_exists"], os.path.join(DATA_DIR, sid + "_sleepscoring.txt"), "txt"),
    ]
    results = []
    for exists, path, file_type in paths:
        try:
            if not exists:
                raise FileNotFoundError(path)
            if file_type == "edf":
                f = pyedflib.EdfReader(path)
                f.close()
            else:
                read_scoring(path)
            results.append(True)
        except Exception:
            results.append(False)

    signal_readable.append(results[0])
    scoring_edf_readable.append(results[1])
    scoring_txt_readable.append(results[2])

recording_inventory["signal_readable"] = signal_readable
recording_inventory["scoring_edf_readable"] = scoring_edf_readable
recording_inventory["scoring_txt_readable"] = scoring_txt_readable

file_integrity_summary = pd.DataFrame({
    "check": ["total_subject_ids", "signal_missing", "scoring_edf_missing",
              "scoring_txt_missing", "signal_unreadable", "scoring_edf_unreadable",
              "scoring_txt_unreadable"],
    "count": [
        len(recording_inventory),
        (~recording_inventory["signal_exists"]).sum(),
        (~recording_inventory["scoring_edf_exists"]).sum(),
        (~recording_inventory["scoring_txt_exists"]).sum(),
        (recording_inventory["signal_exists"] & ~recording_inventory["signal_readable"]).sum(),
        (recording_inventory["scoring_edf_exists"] & ~recording_inventory["scoring_edf_readable"]).sum(),
        (recording_inventory["scoring_txt_exists"] & ~recording_inventory["scoring_txt_readable"]).sum(),
    ]
})

recording_inventory.to_csv(os.path.join(EDA_OUTPUT_DIR, "recording_inventory.csv"), index=False)
file_integrity_summary.to_csv(os.path.join(EDA_OUTPUT_DIR, "file_integrity_summary.csv"), index=False)

print(file_integrity_summary.to_string(index=False))
display(recording_inventory.head(10))

## Section 3: Signal Channel Quality

In [ ]:
readable_ids = recording_inventory[recording_inventory["signal_readable"]]["subject_id"].tolist()

channel_rows = []
for sid in readable_ids:
    f = pyedflib.EdfReader(os.path.join(DATA_DIR, sid + ".edf"))
    labels = f.getSignalLabels()
    rates = f.getSampleFrequencies()
    counts = f.getNSamples()
    f.close()
    for i in range(len(labels)):
        channel_rows.append({
            "subject_id": sid,
            "channel": labels[i],
            "sampling_rate": float(rates[i]),
            "n_samples": int(counts[i]),
            "duration_sec": counts[i] / rates[i],
        })

channel_summary = pd.DataFrame(channel_rows)

n_subjects_read = len(readable_ids)
channel_presence_summary = (
    channel_summary.groupby("channel")["subject_id"].nunique()
    .rename("n_subjects_with_channel")
    .reset_index()
)
channel_presence_summary["n_total_subjects"] = n_subjects_read
channel_presence_summary["percent_present"] = round(100 * channel_presence_summary["n_subjects_with_channel"] / n_subjects_read, 2)

channel_sets = channel_summary.groupby("subject_id")["channel"].apply(lambda x: tuple(sorted(x)))
reference_channels = channel_sets.value_counts().index[0]

consistency_rows = []
for sid, g in channel_summary.groupby("subject_id"):
    consistency_rows.append({
        "subject_id": sid,
        "n_channels": len(g),
        "has_duplicate_channel": bool(g["channel"].duplicated().any()),
        "missing_channels": ", ".join(sorted(set(reference_channels) - set(g["channel"]))),
        "extra_channels": ", ".join(sorted(set(g["channel"]) - set(reference_channels))),
        "unique_sampling_rates": ", ".join(str(r) for r in sorted(g["sampling_rate"].unique())),
        "unique_sample_counts": ", ".join(str(n) for n in sorted(g["n_samples"].unique())),
        "samples_consistent": g["n_samples"].nunique() == 1,
    })

channel_consistency_summary = pd.DataFrame(consistency_rows)

channel_presence_summary.to_csv(os.path.join(EDA_OUTPUT_DIR, "channel_presence_summary.csv"), index=False)
channel_consistency_summary.to_csv(os.path.join(EDA_OUTPUT_DIR, "channel_consistency_summary.csv"), index=False)

print("Reference channel set (most common):")
for c in reference_channels:
    print(" -", c)
print("Subjects with different channel set :", int((channel_consistency_summary["missing_channels"] != "").sum() + (channel_consistency_summary["extra_channels"] != "").sum()))
print("Subjects with duplicate channel     :", int(channel_consistency_summary["has_duplicate_channel"].sum()))
print("Subjects with unequal sample counts :", int((~channel_consistency_summary["samples_consistent"]).sum()))
channel_presence_summary

## Section 4: แสดง Signal จริงของทุก Channel

In [ ]:
def read_all_channels(path):
    f = pyedflib.EdfReader(path)
    names = f.getSignalLabels()
    rates = f.getSampleFrequencies()
    units = [f.getPhysicalDimension(i) for i in range(len(names))]
    data = [f.readSignal(i) for i in range(len(names))]
    f.close()
    return names, rates, data, units

def describe(x):
    return {
        "min": np.min(x),
        "max": np.max(x),
        "mean": np.mean(x),
        "std": np.std(x),
        "median": np.median(x),
        "p1": np.percentile(x, 1),
        "p99": np.percentile(x, 99),
    }

sample_id = "SN001" if "SN001" in readable_ids else readable_ids[0]
sample_names, sample_rates, sample_data, sample_units = read_all_channels(os.path.join(DATA_DIR, sample_id + ".edf"))
print("Sample subject:", sample_id)

for i in range(len(sample_names)):
    print(sample_names[i], "|", sample_rates[i], "Hz |", len(sample_data[i]), "samples |", sample_units[i])

In [ ]:
!pip install mne --quiet

In [ ]:
import mne

print("ค่าตัวอย่างจริงของสัญญาณชีวภาพจาก 10 recordings")

signal_paths = sorted([
    path for path in glob.glob(os.path.join(DATA_DIR, "*.edf"))
    if "_sleepscoring" not in os.path.basename(path).lower()
])

selected_paths = signal_paths[:10]

TARGET_PATTERNS = ["F4-M1", "C4-M1", "O2-M1", "C3-M2", "E1-M2", "E2-M2", "EMG", "ECG"]

sample_dfs = []

for signal_path in selected_paths:
    subject_id = os.path.basename(signal_path).replace(".edf", "")

    raw = mne.io.read_raw_edf(signal_path, preload=False, verbose=False)

    available_channels = []
    for pattern in TARGET_PATTERNS:
        matches = [ch for ch in raw.ch_names if pattern.lower() in ch.lower()]
        if matches:
            available_channels.append(matches[0])

    if len(available_channels) != 8:
        print(f"\n{subject_id}")
        print("Channels ที่พบ:", available_channels)
        print("Channels ทั้งหมด:", raw.ch_names)

    data = raw.get_data(picks=available_channels, start=0, stop=10)
    sfreq = raw.info["sfreq"]

    df = pd.DataFrame(data.T, columns=available_channels)
    df.insert(0, "subject_id", subject_id)
    df.insert(1, "sample_index", np.arange(len(df)))
    df.insert(2, "time_sec", np.arange(len(df)) / sfreq)

    sample_dfs.append(df)

sample_values_10_df = pd.concat(sample_dfs, ignore_index=True)

sample_values_path = os.path.join(EDA_OUTPUT_DIR, "sample_signal_values_10_recordings.csv")
sample_values_10_df.to_csv(sample_values_path, index=False)

print(f"\nบันทึกไฟล์แล้ว: {sample_values_path}")
display(sample_values_10_df)

In [ ]:
window_rows = []

for window_sec in [30, 300]:
    fig, axes = plt.subplots(len(sample_names), 1, figsize=(12, 2.4 * len(sample_names)))
    for i, name in enumerate(sample_names):
        fs = sample_rates[i]
        end_sample = min(int(window_sec * fs), len(sample_data[i]))
        x = sample_data[i][:end_sample]
        t = np.arange(end_sample) / fs
        end_sec = end_sample / fs

        axes[i].plot(t, x, linewidth=0.5)
        axes[i].set_title(f"Raw Signal - {sample_id} - {name}\nTime: 0–{end_sec:g} seconds", fontsize=9)
        axes[i].set_ylabel(sample_units[i])

        row = {
            "subject_id": sample_id,
            "channel": name,
            "unit": sample_units[i],
            "sampling_rate": fs,
            "start_sec": 0,
            "end_sec": end_sec,
            "duration_sec": end_sec,
            "n_samples": end_sample,
        }
        row.update(describe(x))
        window_rows.append(row)

    axes[-1].set_xlabel("Time (seconds)")
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, f"raw_signal_{window_sec}sec_{sample_id}.png"), dpi=120)
    plt.show()

window_stats_df = pd.DataFrame(window_rows)
window_stats_df.to_csv(os.path.join(EDA_OUTPUT_DIR, "sample_window_statistics.csv"), index=False)
window_stats_df

## Section 5: Signal Value Distribution

In [ ]:
n_ch = len(sample_names)
n_rows = (n_ch + 1) // 2
fig, axes = plt.subplots(n_rows, 2, figsize=(12, 3 * n_rows))
axes = axes.flatten()

value_rows = []
for i, name in enumerate(sample_names):
    x = sample_data[i]
    x = x[np.isfinite(x)]
    axes[i].hist(x, bins=100)
    axes[i].set_title(f"{sample_id} - {name}", fontsize=9)
    axes[i].set_xlabel(sample_units[i])

    row = {"subject_id": sample_id, "channel": name, "unit": sample_units[i], "n_samples": len(x)}
    row.update(describe(x))
    value_rows.append(row)

for j in range(n_ch, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, f"value_distribution_{sample_id}.png"), dpi=120)
plt.show()

signal_value_summary = pd.DataFrame(value_rows)
signal_value_summary.to_csv(os.path.join(EDA_OUTPUT_DIR, "signal_value_summary.csv"), index=False)
signal_value_summary

## Section 6: NaN / Inf / Flat Signal Detection

In [ ]:
FLAT_STD_THRESHOLD = 1e-10

quality_rows = []
extreme_rows = []

for sid in readable_ids:
    names, rates, data, units = read_all_channels(os.path.join(DATA_DIR, sid + ".edf"))
    for i, name in enumerate(names):
        x = data[i]
        n_total = len(x)
        n_nan = int(np.isnan(x).sum())
        n_inf = int(np.isinf(x).sum())
        finite = x[np.isfinite(x)]

        if len(finite) > 0:
            std = float(np.std(finite))
            is_flat = std < FLAT_STD_THRESHOLD
        else:
            std = np.nan
            is_flat = True

        quality_rows.append({
            "subject_id": sid,
            "channel": name,
            "n_samples": n_total,
            "n_nan": n_nan,
            "nan_percent": 100 * n_nan / n_total,
            "n_inf": n_inf,
            "inf_percent": 100 * n_inf / n_total,
            "std": std,
            "is_flat": bool(is_flat),
        })

        if len(finite) > 0:
            q1, median, q3 = np.percentile(finite, [25, 50, 75])
            iqr = q3 - q1
            max_value = finite.max()
            min_value = finite.min()
            if iqr > 0:
                extreme_ratio = max(max_value - median, median - min_value) / iqr
            else:
                extreme_ratio = np.nan
            extreme_rows.append({
                "subject_id": sid,
                "channel": name,
                "min": min_value,
                "max": max_value,
                "p1": np.percentile(finite, 1),
                "p99": np.percentile(finite, 99),
                "median": median,
                "iqr": iqr,
                "extreme_ratio": extreme_ratio,
            })

quality_df = pd.DataFrame(quality_rows)

issue_list = []
for _, r in quality_df.iterrows():
    if r["n_nan"] > 0:
        issue_list.append({"subject_id": r["subject_id"], "channel": r["channel"], "issue_type": "nan", "value": r["n_nan"]})
    if r["n_inf"] > 0:
        issue_list.append({"subject_id": r["subject_id"], "channel": r["channel"], "issue_type": "inf", "value": r["n_inf"]})
    if r["is_flat"]:
        issue_list.append({"subject_id": r["subject_id"], "channel": r["channel"], "issue_type": "flat_signal", "value": r["std"]})

signal_quality_issues = pd.DataFrame(issue_list, columns=["subject_id", "channel", "issue_type", "value"])

quality_df.to_csv(os.path.join(EDA_OUTPUT_DIR, "signal_quality_summary.csv"), index=False)
signal_quality_issues.to_csv(os.path.join(EDA_OUTPUT_DIR, "signal_quality_issues.csv"), index=False)

print("Flat signal criterion: std <", FLAT_STD_THRESHOLD)
print("Total NaN  :", int(quality_df["n_nan"].sum()))
print("Total Inf  :", int(quality_df["n_inf"].sum()))
print("Flat signal:", int(quality_df["is_flat"].sum()))
signal_quality_issues.head(20)

## Section 7: Extreme Amplitude / Candidate Outlier Detection

In [ ]:
ROBUST_Z_THRESHOLD = 3.5

extreme_df = pd.DataFrame(extreme_rows)
extreme_df["robust_z"] = np.nan

for channel, idx in extreme_df.groupby("channel").groups.items():
    ratio = extreme_df.loc[idx, "extreme_ratio"]
    median_ratio = ratio.median()
    mad = (ratio - median_ratio).abs().median()
    if mad > 0:
        extreme_df.loc[idx, "robust_z"] = 0.6745 * (ratio - median_ratio) / mad

extreme_df["outlier_indicator"] = extreme_df["robust_z"] > ROBUST_Z_THRESHOLD

extreme_df.to_csv(os.path.join(EDA_OUTPUT_DIR, "candidate_outlier_summary.csv"), index=False)

print("Criterion: robust z of extreme_ratio >", ROBUST_Z_THRESHOLD)
print("extreme_ratio = max(|max - median|, |median - min|) / IQR, compared across subjects within the same channel")
print("Candidate outlier signals:", int(extreme_df["outlier_indicator"].sum()), "จาก", len(extreme_df))

extreme_df[extreme_df["outlier_indicator"]].sort_values("robust_z", ascending=False).head(20)

In [ ]:
candidates = extreme_df[extreme_df["outlier_indicator"]].sort_values("robust_z", ascending=False).head(3)

for _, row in candidates.iterrows():
    names, rates, data, units = read_all_channels(os.path.join(DATA_DIR, row["subject_id"] + ".edf"))
    i = names.index(row["channel"])
    x = data[i]
    fs = rates[i]

    median = np.nanmedian(x)
    peak = int(np.nanargmax(np.abs(x - median)))
    start = max(0, peak - int(15 * fs))
    end = min(len(x), peak + int(15 * fs))

    plt.figure(figsize=(12, 3))
    plt.plot(np.arange(start, end) / fs, x[start:end], linewidth=0.5)
    plt.title(f"Candidate outlier - {row['subject_id']} - {row['channel']} (peak at {peak / fs:.1f} sec)")
    plt.xlabel("Time (seconds)")
    plt.ylabel(units[i])
    plt.tight_layout()
    plt.show()

## Section 8: Annotation / Label Quality

In [ ]:
frames = []
for _, row in recording_inventory[recording_inventory["scoring_txt_readable"]].iterrows():
    sid = row["subject_id"]
    path = os.path.join(DATA_DIR, f"{sid}_sleepscoring.txt")
    df = read_scoring(path)[["Recording onset", "Duration", "Annotation"]].copy()
    df.columns = ["onset", "duration", "annotation"]
    df.insert(0, "subject_id", sid)
    frames.append(df)

all_annotations = pd.concat(frames, ignore_index=True)
all_annotations["annotation"] = all_annotations["annotation"].astype(str).str.strip()
all_annotations["onset"] = pd.to_numeric(all_annotations["onset"], errors="coerce")
all_annotations["duration"] = pd.to_numeric(all_annotations["duration"], errors="coerce")

print("Total annotations:", len(all_annotations))
print("Invalid onset/duration:", int(all_annotations[["onset", "duration"]].isna().any(axis=1).sum()))

annotation_summary = all_annotations["annotation"].value_counts().rename_axis("annotation").reset_index(name="count")
annotation_summary["percentage"] = (100 * annotation_summary["count"] / len(all_annotations)).round(4)
annotation_summary.to_csv(os.path.join(EDA_OUTPUT_DIR, "annotation_summary.csv"), index=False)
display(annotation_summary)

In [ ]:
VALID_STAGES = ["W", "N1", "N2", "N3", "R"]

is_stage = all_annotations["annotation"].str.startswith("Sleep stage")
stage_annotations = all_annotations[is_stage].copy()
other_annotations = all_annotations[~is_stage].copy()
stage_annotations["label"] = stage_annotations["annotation"].str.replace("Sleep stage", "", regex=False).str.strip()

label_distribution = stage_annotations["label"].value_counts().rename_axis("label").reset_index(name="count")
label_distribution["percentage"] = round(100 * label_distribution["count"] / len(stage_annotations), 4)
label_distribution["is_valid_label"] = label_distribution["label"].isin(VALID_STAGES)
label_distribution.to_csv(os.path.join(EDA_OUTPUT_DIR, "label_distribution.csv"), index=False)

n_invalid_labels = int((~stage_annotations["label"].isin(VALID_STAGES)).sum())

print("Sleep stage annotations :", len(stage_annotations))
print("Other annotations       :", len(other_annotations))
print("Invalid / unknown labels:", n_invalid_labels)
print("\nOther annotation types:")
print(other_annotations["annotation"].value_counts())
label_distribution

## Section 9: Epoch Integrity และ Signal–Label Alignment

In [ ]:
EPOCH_SEC = 30

signal_duration = channel_summary.groupby("subject_id")["duration_sec"].min()

print("Sampling rates found (Hz)   :", sorted(channel_summary["sampling_rate"].unique()))
print("Samples per 30-sec epoch    :", sorted((channel_summary["sampling_rate"] * EPOCH_SEC).unique()))

alignment_rows = []
issue_rows = []

def add_issues(sid, g, mask, issue_type):
    for k in np.where(mask)[0]:
        issue_rows.append({
            "subject_id": sid,
            "row_in_sequence": int(k),
            "onset": g["onset"][k],
            "duration": g["duration"][k],
            "label": g["label"][k],
            "issue_type": issue_type,
        })

for sid, g in stage_annotations.groupby("subject_id"):
    if sid not in signal_duration.index:
        continue

    g = g.sort_values("onset").reset_index(drop=True)
    ends = g["onset"] + g["duration"]
    channels = channel_summary[channel_summary["subject_id"] == sid]

    not_30 = (g["duration"] != EPOCH_SEC).values
    misaligned = (np.abs(np.round(g["onset"] / EPOCH_SEC) * EPOCH_SEC - g["onset"]) > 1e-6).values
    out_of_range = (ends > signal_duration[sid] + 1e-6).values

    unmappable = np.zeros(len(g), dtype=bool)
    for _, c in channels.iterrows():
        start_idx = g["onset"].values * c["sampling_rate"]
        end_idx = ends.values * c["sampling_rate"]
        unmappable = unmappable | (np.abs(start_idx - np.round(start_idx)) > 1e-6) | (end_idx > c["n_samples"] + 1e-6)

    gap = np.r_[0.0, g["onset"].values[1:] - ends.values[:-1]]
    has_gap = gap > 1e-6
    has_overlap = gap < -1e-6

    add_issues(sid, g, not_30, "duration_not_30")
    add_issues(sid, g, misaligned, "onset_not_multiple_of_30")
    add_issues(sid, g, out_of_range, "outside_signal")
    add_issues(sid, g, unmappable, "cannot_map_to_samples")
    add_issues(sid, g, has_gap, "gap_before_epoch")
    add_issues(sid, g, has_overlap, "overlap_with_previous")

    alignment_rows.append({
        "subject_id": sid,
        "n_stage_epochs": len(g),
        "signal_duration_sec": signal_duration[sid],
        "last_epoch_end_sec": ends.max(),
        "epoch_samples_integer": bool(((channels["sampling_rate"] * EPOCH_SEC) % 1 == 0).all()),
        "n_duration_not_30": int(not_30.sum()),
        "n_onset_misaligned": int(misaligned.sum()),
        "n_outside_signal": int(out_of_range.sum()),
        "n_cannot_map_to_samples": int(unmappable.sum()),
        "n_gap": int(has_gap.sum()),
        "n_overlap": int(has_overlap.sum()),
    })

epoch_alignment_summary = pd.DataFrame(alignment_rows)
issue_columns = ["subject_id", "row_in_sequence", "onset", "duration", "label", "issue_type"]
epoch_alignment_issues = pd.DataFrame(issue_rows, columns=issue_columns)

epoch_alignment_summary.to_csv(os.path.join(EDA_OUTPUT_DIR, "epoch_alignment_summary.csv"), index=False)
epoch_alignment_issues.to_csv(os.path.join(EDA_OUTPUT_DIR, "epoch_alignment_issues.csv"), index=False)

print("Total alignment issues:", len(epoch_alignment_issues))
print(epoch_alignment_issues["issue_type"].value_counts())
epoch_alignment_summary.head(10)

## Section 10: ตรวจสอบ Sleep Stage Sequence ของแต่ละ Subject

In [ ]:
MIN_RUN = 5

run_rows = []

for sid, g in stage_annotations.groupby("subject_id"):
    g = g[g["label"].isin(VALID_STAGES)].sort_values("onset").reset_index(drop=True)
    start = 0
    for k in range(1, len(g) + 1):
        new_run = (
            k == len(g)
            or g["label"][k] != g["label"][k - 1]
            or abs(g["onset"][k] - g["onset"][k - 1] - EPOCH_SEC) > 1e-6
        )
        if new_run:
            run_rows.append({
                "subject_id": sid,
                "stage": g["label"][start],
                "start_epoch": int(round(g["onset"][start] / EPOCH_SEC)),
                "end_epoch": int(round(g["onset"][k - 1] / EPOCH_SEC)),
                "n_epochs": k - start,
                "start_time_sec": g["onset"][start],
                "end_time_sec": g["onset"][k - 1] + EPOCH_SEC,
            })
            start = k

runs_df = pd.DataFrame(run_rows)
runs_df.to_csv(os.path.join(EDA_OUTPUT_DIR, "stage_runs_all.csv"), index=False)

scored_subjects = recording_inventory.loc[recording_inventory["scoring_txt_readable"], "subject_id"].tolist()

max_run = (
    runs_df.groupby(["subject_id", "stage"])["n_epochs"].max()
    .unstack(fill_value=0)
    .reindex(index=scored_subjects, columns=VALID_STAGES, fill_value=0)
    .rename_axis("subject_id")
)

stage_run_summary = max_run.reset_index().melt(id_vars="subject_id", var_name="stage", value_name="max_consecutive_epochs")
stage_run_summary["has_5_consecutive"] = stage_run_summary["max_consecutive_epochs"] >= MIN_RUN
stage_run_summary = stage_run_summary.sort_values(["subject_id", "stage"]).reset_index(drop=True)
stage_run_summary.to_csv(os.path.join(EDA_OUTPUT_DIR, "stage_run_summary.csv"), index=False)

stage_run_summary.head(15)

## Section 11: Dataset Eligibility ตาม Requirement ใหม่

In [ ]:
eligibility = max_run.copy()
eligibility.columns = [f"{s}_max_run" for s in VALID_STAGES]

for s in VALID_STAGES:
    eligibility[f"{s}_has_5"] = eligibility[f"{s}_max_run"] >= MIN_RUN

eligibility["eligible_all_5_stages"] = eligibility[[f"{s}_has_5" for s in VALID_STAGES]].all(axis=1)
eligibility = eligibility.reset_index()
eligibility.to_csv(os.path.join(EDA_OUTPUT_DIR, "subject_stage_eligibility.csv"), index=False)

print("Total Subjects           :", len(eligibility))
for s in VALID_STAGES:
    print(f"Subjects passing {s:<3}     :", int(eligibility[f"{s}_has_5"].sum()))
print("Subjects passing all 5   :", int(eligibility["eligible_all_5_stages"].sum()))

eligibility[~eligibility["eligible_all_5_stages"]]

## Section 12: ตัวอย่าง Stage Sequence พร้อมช่วง Epoch

In [ ]:
eligible_ids = eligibility[eligibility["eligible_all_5_stages"]]["subject_id"].tolist()
not_eligible_ids = eligibility[~eligibility["eligible_all_5_stages"]]["subject_id"].tolist()
longest_run_id = runs_df.sort_values("n_epochs", ascending=False)["subject_id"].iloc[0]

selected_subjects = []
if len(eligible_ids) > 0:
    selected_subjects.append(eligible_ids[0])
if len(not_eligible_ids) > 0:
    selected_subjects.append(not_eligible_ids[0])
if longest_run_id not in selected_subjects:
    selected_subjects.append(longest_run_id)
for sid in eligible_ids + not_eligible_ids:
    if len(selected_subjects) >= 3:
        break
    if sid not in selected_subjects:
        selected_subjects.append(sid)

selected_windows = {}

for sid in selected_subjects:
    print("=" * 50)
    print("Subject:", sid)
    selected_windows[sid] = {}
    sub = runs_df[runs_df["subject_id"] == sid]
    for stage in VALID_STAGES:
        stage_runs = sub[(sub["stage"] == stage) & (sub["n_epochs"] >= MIN_RUN)]
        print()
        if len(stage_runs) == 0:
            print("Stage       :", stage)
            print(f"No run of {MIN_RUN} epochs (max run = {int(max_run.loc[sid, stage])})")
            continue
        r = stage_runs.iloc[0]
        start_epoch = int(r["start_epoch"])
        selected_windows[sid][stage] = start_epoch
        print("Stage       :", stage)
        print("Start Epoch :", start_epoch)
        print("End Epoch   :", start_epoch + MIN_RUN - 1)
        print("Count       :", MIN_RUN)
        print("Start Time  :", r["start_time_sec"], "sec")
        print("End Time    :", r["start_time_sec"] + MIN_RUN * EPOCH_SEC, "sec")
        print("Full run    :", int(r["n_epochs"]), "epochs")

## Section 13: Stage Timeline

In [ ]:
stage_order = ["N3", "N2", "N1", "R", "W"]
stage_y = {s: i for i, s in enumerate(stage_order)}
stage_color = {"W": "tab:orange", "N1": "tab:green", "N2": "tab:blue", "N3": "tab:purple", "R": "tab:red"}

for sid in selected_subjects:
    g = stage_annotations[(stage_annotations["subject_id"] == sid) & (stage_annotations["label"].isin(VALID_STAGES))]
    g = g.sort_values("onset")
    epochs = np.round(g["onset"] / EPOCH_SEC).astype(int)
    y = g["label"].map(stage_y)

    fig, ax = plt.subplots(figsize=(14, 3.5))
    ax.step(epochs, y, where="post", color="gray", linewidth=0.8)

    for stage, start in selected_windows[sid].items():
        ax.axvspan(start - 0.5, start + MIN_RUN - 0.5, color=stage_color[stage], alpha=0.6)
        ax.scatter(range(start, start + MIN_RUN), [stage_y[stage]] * MIN_RUN, color=stage_color[stage], s=40, zorder=3, label=f"{stage} (epoch {start}-{start + MIN_RUN - 1})")

    ax.set_yticks(range(len(stage_order)))
    ax.set_yticklabels(stage_order)
    ax.set_xlabel("Epoch index (1 epoch = 30 sec)")
    ax.set_ylabel("Sleep stage")
    ax.set_title(f"Sleep Stage Timeline - {sid} (highlight = first {MIN_RUN} consecutive epochs per stage)")
    ax.legend(loc="upper right", fontsize=8, ncol=5)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, f"stage_timeline_{sid}.png"), dpi=120)
    plt.show()

## Section 14: Final EDA Summary

In [ ]:
summary = {
    "criteria": {
        "flat_signal": f"std < {FLAT_STD_THRESHOLD}",
        "candidate_outlier": f"robust z of extreme_ratio > {ROBUST_Z_THRESHOLD}",
        "min_consecutive_epochs": MIN_RUN,
        "epoch_sec": EPOCH_SEC,
    },
    "files": {
        "subjects_checked": int(len(recording_inventory)),
        "unreadable_signal": int(file_integrity_summary.loc[file_integrity_summary["check"] == "signal_unreadable", "count"].iloc[0]),
        "unreadable_scoring": int(file_integrity_summary.loc[file_integrity_summary["check"] == "scoring_txt_unreadable", "count"].iloc[0]),
        "duplicate_subjects": int(recording_inventory["subject_id"].duplicated().sum()),
        "orphan_scoring": int(len(orphan_scoring)),
        "orphan_signal": int(len(orphan_signal)),
    },
    "signal_quality": {
        "channels_found": sorted(channel_summary["channel"].unique().tolist()),
        "subjects_with_different_channel_set": int(((channel_consistency_summary["missing_channels"] != "") | (channel_consistency_summary["extra_channels"] != "")).sum()),
        "subjects_with_unequal_sample_counts": int((~channel_consistency_summary["samples_consistent"]).sum()),
        "total_nan": int(quality_df["n_nan"].sum()),
        "total_inf": int(quality_df["n_inf"].sum()),
        "flat_signals": int(quality_df["is_flat"].sum()),
        "candidate_outlier_signals": int(extreme_df["outlier_indicator"].sum()),
    },
    "labels": {
        "annotation_types": annotation_summary["annotation"].tolist(),
        "invalid_or_unknown_labels": n_invalid_labels,
        "label_counts": {r["label"]: int(r["count"]) for _, r in label_distribution.iterrows()},
        "alignment_issues": int(len(epoch_alignment_issues)),
        "subjects_with_alignment_issue": int(epoch_alignment_issues["subject_id"].nunique()),
    },
    "eligibility": {
        "total_subjects": int(len(eligibility)),
        **{f"subjects_with_{s}_ge_5": int(eligibility[f"{s}_has_5"].sum()) for s in VALID_STAGES},
        "subjects_eligible_all_5_stages": int(eligibility["eligible_all_5_stages"].sum()),
    },
}

with open(os.path.join(EDA_OUTPUT_DIR, "eda_summary.json"), "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("=== DATA QUALITY SUMMARY ===")
print()
print("Subjects checked             :", summary["files"]["subjects_checked"])
print("Unreadable recordings        :", summary["files"]["unreadable_signal"], "(signal),", summary["files"]["unreadable_scoring"], "(scoring)")
print("Duplicate subjects           :", summary["files"]["duplicate_subjects"])
print()
print("NaN values                   :", summary["signal_quality"]["total_nan"])
print("Inf values                   :", summary["signal_quality"]["total_inf"])
print("Flat signals                 :", summary["signal_quality"]["flat_signals"])
print("Candidate outlier signals    :", summary["signal_quality"]["candidate_outlier_signals"])
print()
print("Invalid / unknown labels     :", summary["labels"]["invalid_or_unknown_labels"])
print("Signal-label alignment issue :", summary["labels"]["alignment_issues"], "issues in", summary["labels"]["subjects_with_alignment_issue"], "subjects")
print()
for s in VALID_STAGES:
    print(f"Subjects with {s:<3} >= 5        :", summary["eligibility"][f"subjects_with_{s}_ge_5"])
print()
print("Subjects eligible for all 5 stages:", summary["eligibility"]["subjects_eligible_all_5_stages"])